# 02 — Check Environment & Terrain Outputs (Phase 2)

Quick visual / numerical sanity check of the GeoTIFFs produced by `scripts/task1/phase_02_environment_terrain/extract_environment_terrain.py`.

Expected inputs in `data/raw/environment_terrain/`:
- `<fire_id>__terrain__static.tif` (SRTM, 30 m)
- `<fire_id>__weather__stacked.tif` (ERA5-Land, ~9 km, all days in one multi-band file)

Legacy daily mode (if used): `<fire_id>__weather__YYYY-MM-DD.tif`

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt

ENV_DIR = Path('../data/raw/environment_terrain')
sorted(ENV_DIR.glob('*.tif'))

In [ ]:
FIRE_ID = 'REPLACE_WITH_FIRE_ID'

terrain_path = ENV_DIR / f'{FIRE_ID}__terrain__static.tif'
stacked_weather_path = ENV_DIR / f'{FIRE_ID}__weather__stacked.tif'
daily_weather_paths = sorted(
    p for p in ENV_DIR.glob(f'{FIRE_ID}__weather__*.tif')
    if p.name != stacked_weather_path.name
)

print('Terrain :', terrain_path, terrain_path.exists())
print('Stacked weather :', stacked_weather_path, stacked_weather_path.exists())
print('Daily weather files (legacy) :', len(daily_weather_paths))
for p in daily_weather_paths[:5]:
    print('  -', p.name)

In [ ]:
# Inspect terrain bands (elevation / slope / aspect).
with rasterio.open(terrain_path) as src:
    print('Terrain metadata:')
    print('  CRS    :', src.crs)
    print('  Shape  :', (src.height, src.width))
    print('  Bands  :', src.descriptions)
    terrain = src.read()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, band, name in zip(axes, terrain, ['elevation', 'slope', 'aspect']):
    im = ax.imshow(band, cmap='terrain' if name == 'elevation' else 'viridis')
    ax.set_title(name)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect stacked weather (default) or one daily file (legacy).
weather_path = stacked_weather_path if stacked_weather_path.exists() else (
    daily_weather_paths[0] if daily_weather_paths else None
)

if weather_path:
    with rasterio.open(weather_path) as src:
        print('Weather metadata for', weather_path.name)
        print('  CRS    :', src.crs)
        print('  Shape  :', (src.height, src.width))
        print('  Band count :', src.count)
        print('  First 6 bands :', list(src.descriptions or [])[:6])
        data = src.read()

    # Plot first day (first 6 bands) if stacked; otherwise plot all bands.
    bands_to_plot = min(6, data.shape[0])
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.ravel()
    for ax, idx in zip(axes, range(bands_to_plot)):
        im = ax.imshow(data[idx])
        label = (src.descriptions or [f'band_{i+1}' for i in range(src.count)])[idx]
        ax.set_title(label)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
else:
    print('No weather files for', FIRE_ID)

## TODO
- Loop over all fires and print a coverage table (days available per fire).
- Spot-check temperature / RH / wind ranges for plausibility.